# <center>Práctica 07: 3D Scatter Plot con Sprites de Pokémon</center>


**Nombre del estudiante:** Al Farias Leyva
**Grupo:** 9A IDGS 
**Fecha:** 06 de agosto de 2026  
**Asignatura:** ECBD  

### Objetivo
Analizar las estadísticas base de los Pokémon de las generaciones 1 a 9 y construir una visualización tridimensional interactiva que permita comparar generación, tipo principal y promedio de estadísticas, diferenciando los puntos por tipo e integrando el sprite correspondiente mediante eventos interactivos y filtros.


## Contenido del Notebook

1. Importación de librerías.
2. Carga y descripción del dataset.
3. Inspección inicial.
4. Limpieza y normalización.
5. Selección de variables y promedio de estadísticas.
6. Análisis estadístico descriptivo.
7. Preparación de generación, tipo principal y sprites.
8. Primera versión del Scatter Plot 3D.
9. Visualización final con colores, hover, sprites y filtros.
10. Hallazgos, exportación HTML y conclusiones.

## 1. Importación de librerías — 2 firmas

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, IFrame, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librerías importadas correctamente.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Librerías importadas correctamente.
Pandas: 2.3.3
NumPy: 2.3.5


## 2. Carga y descripción del dataset — 3 firmas

El archivo **Pokemon.csv** procede del repositorio público `lgreski/pokemonData`. El repositorio reúne estadísticas básicas de Pokémon obtenidas de PokémonDB y contiene datos de las generaciones 1 a 9. El CSV consolidado incluye número de Pokédex, nombre, forma, tipos, total, HP, ataque, defensa, ataque especial, defensa especial, velocidad y generación.

La práctica intenta leer primero el archivo local para que el Notebook sea reproducible. Si no se encuentra, utiliza la URL *raw* del repositorio.

In [2]:
CSV_URL = "https://raw.githubusercontent.com/lgreski/pokemonData/master/Pokemon.csv"
RUTAS_LOCALES = [Path("Pokemon.csv"), Path("Practica07/Pokemon.csv")]
CSV_LOCAL = next((ruta for ruta in RUTAS_LOCALES if ruta.exists()), None)
FUENTE_CSV = CSV_LOCAL if CSV_LOCAL is not None else CSV_URL

pokemon_original = pd.read_csv(FUENTE_CSV, keep_default_na=False)

print(f"Dataset cargado desde: {FUENTE_CSV}")
print(f"Registros cargados: {len(pokemon_original):,}")

print("\nColumnas:")
print(pokemon_original.columns.tolist())

pokemon_original.head()

Dataset cargado desde: Pokemon.csv
Registros cargados: 1,215

Columnas:
['ID', 'Name', 'Form', 'Type1', 'Type2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation']


,ID,Name,Form,Type1,Type2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,1,Bulbasaur,,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,,Fire,,309,39,52,43,60,50,65,1
4,5,Charmeleon,,Fire,,405,58,64,58,80,65,80,1


## 3. Inspección inicial del dataset — 2 firmas

### Primeros registros: `head()`

In [3]:
pokemon_original.head(10)

,ID,Name,Form,Type1,Type2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,1,Bulbasaur,,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,,Fire,,309,39,52,43,60,50,65,1
4,5,Charmeleon,,Fire,,405,58,64,58,80,65,80,1
5,6,Charizard,,Fire,Flying,534,78,84,78,109,85,100,1
6,7,Squirtle,,Water,,314,44,48,65,50,64,43,1
7,8,Wartortle,,Water,,405,59,63,80,65,80,58,1
8,9,Blastoise,,Water,,530,79,83,100,85,105,78,1
9,10,Caterpie,,Bug,,195,45,30,35,20,20,45,1


### Dimensiones: `shape`

In [4]:
filas, columnas = pokemon_original.shape
print(f"El dataset contiene {filas:,} filas y {columnas} columnas.")
pokemon_original.shape

El dataset contiene 1,215 filas y 13 columnas.


(1215, 13)

### Nombres de columnas

In [5]:
pokemon_original.columns.tolist()

['ID',
 'Name',
 'Form',
 'Type1',
 'Type2',
 'Total',
 'HP',
 'Attack',
 'Defense',
 'Sp. Atk',
 'Sp. Def',
 'Speed',
 'Generation']

### Tipos de datos y valores no nulos: `info()`

In [6]:
pokemon_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          1215 non-null   int64 
 1   Name        1215 non-null   object
 2   Form        1215 non-null   object
 3   Type1       1215 non-null   object
 4   Type2       1215 non-null   object
 5   Total       1215 non-null   int64 
 6   HP          1215 non-null   int64 
 7   Attack      1215 non-null   int64 
 8   Defense     1215 non-null   int64 
 9   Sp. Atk     1215 non-null   int64 
 10  Sp. Def     1215 non-null   int64 
 11  Speed       1215 non-null   int64 
 12  Generation  1215 non-null   int64 
dtypes: int64(9), object(4)
memory usage: 123.5+ KB


### Resumen numérico: `describe()`

In [7]:
pokemon_original.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,"1,215.00",501.74,298.98,1.00,240.50,495.00,753.50,"1,025.00"
Total,"1,215.00",443.10,121.19,175.00,332.00,465.00,521.00,"1,125.00"
HP,"1,215.00",71.24,26.93,1.00,52.00,70.00,85.00,255.00
Attack,"1,215.00",81.15,32.04,5.00,57.00,80.00,100.00,190.00
Defense,"1,215.00",75.01,30.74,5.00,52.00,70.00,91.00,250.00
Sp. Atk,"1,215.00",73.22,32.76,10.00,50.00,65.00,95.00,194.00
Sp. Def,"1,215.00",72.44,27.58,20.00,51.00,70.00,90.00,250.00
Speed,"1,215.00",70.03,30.16,5.00,45.00,68.00,91.00,200.00
Generation,"1,215.00",5.06,2.60,1.00,3.00,5.00,7.00,9.00


## 4. Limpieza y normalización — 5 firmas

Se normalizarán los nombres de columnas a formato `snake_case`, se eliminarán espacios sobrantes, se estandarizarán los tipos con formato de título y se convertirán las variables estadísticas a valores numéricos. Los campos vacíos de forma y segundo tipo no se consideran errores: representan la forma base y a los Pokémon de un solo tipo.

In [8]:
# Reporte antes de la limpieza
reporte_antes = pd.Series({
    "filas": len(pokemon_original),
    "columnas": pokemon_original.shape[1],
    "nulos_reales": int(pokemon_original.isna().sum().sum()),
    "celdas_vacias": int(pokemon_original.astype(str).apply(lambda c: c.str.strip().eq("")).sum().sum()),
    "duplicados_exactos": int(pokemon_original.duplicated().sum()),
})
reporte_antes.to_frame("antes_de_limpieza")

,antes_de_limpieza
filas,1215
columnas,13
nulos_reales,0
celdas_vacias,1531
duplicados_exactos,0


In [9]:
# Copia de trabajo
pokemon = pokemon_original.copy(deep=True)

# Normalización de columnas
renombrar_columnas = {
    "ID": "id",
    "Name": "nombre",
    "Form": "forma",
    "Type1": "tipo_1",
    "Type2": "tipo_2",
    "Total": "total",
    "HP": "hp",
    "Attack": "ataque",
    "Defense": "defensa",
    "Sp. Atk": "ataque_especial",
    "Sp. Def": "defensa_especial",
    "Speed": "velocidad",
    "Generation": "generacion",
}
pokemon = pokemon.rename(columns=renombrar_columnas)

# Normalización de texto
columnas_texto = ["nombre", "forma", "tipo_1", "tipo_2"]
for columna in columnas_texto:
    pokemon[columna] = pokemon[columna].astype(str).str.strip()

pokemon["nombre"] = pokemon["nombre"].str.replace(r"\s+", " ", regex=True)
pokemon["forma"] = pokemon["forma"].replace("", "Forma base")
pokemon["tipo_2"] = pokemon["tipo_2"].replace("", "Sin tipo secundario")
pokemon["tipo_1"] = pokemon["tipo_1"].str.title()
pokemon["tipo_2"] = pokemon["tipo_2"].where(
    pokemon["tipo_2"].eq("Sin tipo secundario"),
    pokemon["tipo_2"].str.title()
)

# Conversión numérica
columnas_numericas = [
    "id", "total", "hp", "ataque", "defensa",
    "ataque_especial", "defensa_especial", "velocidad", "generacion"
]
for columna in columnas_numericas:
    pokemon[columna] = pd.to_numeric(pokemon[columna], errors="coerce")

# Eliminación de duplicados exactos
pokemon = pokemon.drop_duplicates().copy()

# Tratamiento de valores inválidos
estadisticas = ["hp", "ataque", "defensa", "ataque_especial", "defensa_especial", "velocidad"]
mascara_valida = (
    pokemon["id"].between(1, 1025)
    & pokemon["generacion"].between(1, 9)
    & pokemon[estadisticas].notna().all(axis=1)
    & pokemon[estadisticas].ge(0).all(axis=1)
    & pokemon["nombre"].ne("")
    & pokemon["tipo_1"].ne("")
)
pokemon = pokemon.loc[mascara_valida].reset_index(drop=True)

# Tipos enteros
pokemon[columnas_numericas] = pokemon[columnas_numericas].astype(int)

print("Limpieza terminada.")
pokemon.head()

Limpieza terminada.


,id,nombre,forma,tipo_1,tipo_2,total,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,generacion
0,1,Bulbasaur,Forma base,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,Forma base,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,Forma base,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,Forma base,Fire,Sin tipo secundario,309,39,52,43,60,50,65,1
4,5,Charmeleon,Forma base,Fire,Sin tipo secundario,405,58,64,58,80,65,80,1


In [10]:
# Reporte comparativo antes y después
reporte_despues = pd.Series({
    "filas": len(pokemon),
    "columnas": pokemon.shape[1],
    "nulos_reales": int(pokemon.isna().sum().sum()),
    "celdas_vacias": int(pokemon.astype(str).apply(lambda c: c.str.strip().eq("")).sum().sum()),
    "duplicados_exactos": int(pokemon.duplicated().sum()),
})

reporte_limpieza = pd.concat(
    [reporte_antes.rename("antes"), reporte_despues.rename("despues")],
    axis=1
)
reporte_limpieza

,antes,despues
filas,1215,1215
columnas,13,13
nulos_reales,0,0
celdas_vacias,1531,0
duplicados_exactos,0,0


In [11]:
# Verificación de posibles datos incorrectos después de limpiar
verificacion = pd.Series({
    "generaciones_fuera_de_1_a_9": int((~pokemon["generacion"].between(1, 9)).sum()),
    "ids_fuera_de_1_a_1025": int((~pokemon["id"].between(1, 1025)).sum()),
    "estadisticas_negativas": int((pokemon[estadisticas] < 0).sum().sum()),
    "tipos_principales_vacios": int(pokemon["tipo_1"].eq("").sum()),
})
verificacion.to_frame("cantidad")

,cantidad
generaciones_fuera_de_1_a_9,0
ids_fuera_de_1_a_1025,0
estadisticas_negativas,0
tipos_principales_vacios,0


## 5. Selección y justificación de variables — 3 firmas

Se seleccionan **HP, ataque, defensa, ataque especial, defensa especial y velocidad** porque representan las seis estadísticas base que describen el desempeño general de cada Pokémon:

- **HP:** capacidad para resistir daño.
- **Ataque y defensa:** desempeño físico ofensivo y defensivo.
- **Ataque especial y defensa especial:** desempeño especial ofensivo y defensivo.
- **Velocidad:** prioridad relativa para actuar.

La media de estas seis variables permite construir una medida resumida comparable entre Pokémon sin favorecer una sola estadística.

In [12]:
variables_estadisticas = [
    "hp", "ataque", "defensa",
    "ataque_especial", "defensa_especial", "velocidad"
]

pokemon["promedio_estadisticas"] = pokemon[variables_estadisticas].mean(axis=1).round(2)
pokemon[["id", "nombre", "forma", *variables_estadisticas, "promedio_estadisticas"]].head(10)

,id,nombre,forma,hp,ataque,defensa,ataque_especial,defensa_especial,velocidad,promedio_estadisticas
0,1,Bulbasaur,Forma base,45,49,49,65,65,45,53.00
1,2,Ivysaur,Forma base,60,62,63,80,80,60,67.50
2,3,Venusaur,Forma base,80,82,83,100,100,80,87.50
3,4,Charmander,Forma base,39,52,43,60,50,65,51.50
4,5,Charmeleon,Forma base,58,64,58,80,65,80,67.50
5,6,Charizard,Forma base,78,84,78,109,85,100,89.00
6,7,Squirtle,Forma base,44,48,65,50,64,43,52.33
7,8,Wartortle,Forma base,59,63,80,65,80,58,67.50
8,9,Blastoise,Forma base,79,83,100,85,105,78,88.33
9,10,Caterpie,Forma base,45,30,35,20,20,45,32.50


## 6. Análisis estadístico descriptivo — 4 firmas

In [13]:
analisis_descriptivo = pokemon[variables_estadisticas + ["promedio_estadisticas"]].agg(
    ["mean", "median", "min", "max", "std"]
).T
analisis_descriptivo.index.name = "variable"
analisis_descriptivo.round(2)

,mean,median,min,max,std
variable,,,,,
hp,71.24,70.00,1.00,255.00,26.93
ataque,81.15,80.00,5.00,190.00,32.04
defensa,75.01,70.00,5.00,250.00,30.74
ataque_especial,73.22,65.00,10.00,194.00,32.76
defensa_especial,72.44,70.00,20.00,250.00,27.58
velocidad,70.03,68.00,5.00,200.00,30.16
promedio_estadisticas,73.85,77.50,29.17,187.50,20.20


In [14]:
# Visualización complementaria de la distribución de estadísticas
estadisticas_largas = pokemon[variables_estadisticas].melt(
    var_name="estadistica",
    value_name="valor"
)
fig_cajas = px.box(
    estadisticas_largas,
    x="estadistica",
    y="valor",
    points=False,
    title="Distribución de las seis estadísticas base"
)
fig_cajas.update_layout(xaxis_title="Estadística", yaxis_title="Valor base", height=500)
fig_cajas.show()

## 7. Preparación de generación y tipo principal — 3 firmas

El CSV contiene formas alternativas y, por ello, puede presentar varias filas con el mismo número de Pokédex. Para relacionar cada punto con un sprite nacional inequívoco, la visualización principal usa un solo registro por `id`: se prioriza la **Forma base** y, cuando no existe, se conserva la primera forma disponible. El análisis descriptivo anterior sí conserva todos los registros válidos.

In [15]:
# Priorización de la forma base por cada número de Pokédex
pokemon_visualizacion = (
    pokemon.assign(es_forma_base=pokemon["forma"].eq("Forma base"))
    .sort_values(["id", "es_forma_base", "forma"], ascending=[True, False, True])
    .drop_duplicates(subset="id", keep="first")
    .drop(columns="es_forma_base")
    .reset_index(drop=True)
)

orden_tipos = [
    "Normal", "Fire", "Water", "Electric", "Grass", "Ice",
    "Fighting", "Poison", "Ground", "Flying", "Psychic", "Bug",
    "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy"
]

pokemon_visualizacion["tipo_1"] = pd.Categorical(
    pokemon_visualizacion["tipo_1"],
    categories=orden_tipos,
    ordered=True
)
pokemon_visualizacion["tipo_codigo"] = pokemon_visualizacion["tipo_1"].cat.codes
pokemon_visualizacion = pokemon_visualizacion.sort_values(
    ["generacion", "tipo_codigo", "promedio_estadisticas", "id"]
).reset_index(drop=True)

print(f"Registros totales limpios (incluidas formas): {len(pokemon):,}")
print(f"Pokémon únicos usados en la visualización: {len(pokemon_visualizacion):,}")
print(f"Generaciones: {sorted(pokemon_visualizacion['generacion'].unique())}")
pokemon_visualizacion[["id", "nombre", "forma", "tipo_1", "generacion", "tipo_codigo"]].head(12)

Registros totales limpios (incluidas formas): 1,215
Pokémon únicos usados en la visualización: 1,025
Generaciones: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


,id,nombre,forma,tipo_1,generacion,tipo_codigo
0,16,Pidgey,Forma base,Normal,1,0
1,19,Rattata,Forma base,Normal,1,0
2,21,Spearow,Forma base,Normal,1,0
3,39,Jigglypuff,Forma base,Normal,1,0
4,132,Ditto,Forma base,Normal,1,0
5,52,Meowth,Forma base,Normal,1,0
6,84,Doduo,Forma base,Normal,1,0
7,133,Eevee,Forma base,Normal,1,0
8,17,Pidgeotto,Forma base,Normal,1,0
9,83,Farfetch'd,Forma base,Normal,1,0


## 8. Obtención, relación y validación de sprites — 2 firmas

Los sprites se relacionan con el número de la Pokédex mediante el repositorio público de sprites de PokeAPI. La ruta utilizada sigue el patrón:

`https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{id}.png`

La validación comprueba que cada URL tenga el patrón esperado, que el `id` esté entre 1 y 1025 y que no existan rutas duplicadas para IDs distintos.

In [16]:
SPRITE_BASE = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon"
pokemon_visualizacion["sprite_url"] = pokemon_visualizacion["id"].map(
    lambda numero: f"{SPRITE_BASE}/{numero}.png"
)

patron_sprite = re.compile(
    r"^https://raw\.githubusercontent\.com/PokeAPI/sprites/master/sprites/pokemon/\d+\.png$"
)
pokemon_visualizacion["sprite_url_valida"] = pokemon_visualizacion["sprite_url"].map(
    lambda url: bool(patron_sprite.match(url))
)

validacion_sprites = pd.Series({
    "rutas_generadas": len(pokemon_visualizacion),
    "rutas_con_formato_valido": int(pokemon_visualizacion["sprite_url_valida"].sum()),
    "rutas_con_formato_invalido": int((~pokemon_visualizacion["sprite_url_valida"]).sum()),
    "ids_fuera_de_rango": int((~pokemon_visualizacion["id"].between(1, 1025)).sum()),
    "urls_duplicadas": int(pokemon_visualizacion["sprite_url"].duplicated().sum()),
})
validacion_sprites.to_frame("resultado")

,resultado
rutas_generadas,1025
rutas_con_formato_valido,1025
rutas_con_formato_invalido,0
ids_fuera_de_rango,0
urls_duplicadas,0


In [17]:
# Muestra visual de sprites relacionados con los primeros Pokémon
muestra_sprites = pokemon_visualizacion.head(12)
tarjetas = "".join(
    f"""
    <div style='display:inline-block;width:120px;text-align:center;margin:6px;padding:8px;
                border:1px solid #ddd;border-radius:10px;background:white'>
        <img src='{fila.sprite_url}' width='72' height='72' loading='lazy'><br>
        <b>#{fila.id}</b><br>{fila.nombre}
    </div>
    """
    for fila in muestra_sprites.itertuples()
)
display(HTML(f"<div>{tarjetas}</div>"))

## 9. Primera versión del Scatter Plot 3D — 3 firmas

Los ejes se definen de esta forma:

- **X:** generación.
- **Y:** tipo principal codificado y etiquetado con su nombre.
- **Z:** promedio de las seis estadísticas base.

In [18]:
fig_primera = px.scatter_3d(
    pokemon_visualizacion,
    x="generacion",
    y="tipo_codigo",
    z="promedio_estadisticas",
    color="tipo_1",
    hover_name="nombre",
    custom_data=["id", "forma", "tipo_1", "generacion", "promedio_estadisticas"],
    title="Primera versión: Pokémon por generación, tipo y promedio de estadísticas",
    opacity=0.72,
    height=760
)

fig_primera.update_traces(marker={"size": 4})
fig_primera.update_layout(
    scene={
        "xaxis": {"title": "Generación", "dtick": 1},
        "yaxis": {
            "title": "Tipo principal",
            "tickmode": "array",
            "tickvals": list(range(len(orden_tipos))),
            "ticktext": orden_tipos,
        },
        "zaxis": {"title": "Promedio de estadísticas"},
    }
)
fig_primera.show()

## 10. Diseño final: color, símbolos, hover, cámara y escala — 8 firmas

La versión final crea una traza por tipo principal, asigna un color representativo, configura la información emergente y personaliza la cámara, opacidad, leyenda, etiquetas y tamaño de los puntos.

In [19]:
colores_tipo = {
    "Normal": "#A8A77A", "Fire": "#EE8130", "Water": "#6390F0",
    "Electric": "#F7D02C", "Grass": "#7AC74C", "Ice": "#96D9D6",
    "Fighting": "#C22E28", "Poison": "#A33EA1", "Ground": "#E2BF65",
    "Flying": "#A98FF3", "Psychic": "#F95587", "Bug": "#A6B91A",
    "Rock": "#B6A136", "Ghost": "#735797", "Dragon": "#6F35FC",
    "Dark": "#705746", "Steel": "#B7B7CE", "Fairy": "#D685AD",
}

fig_final = go.Figure()

for tipo in orden_tipos:
    subconjunto = pokemon_visualizacion[pokemon_visualizacion["tipo_1"] == tipo]
    if subconjunto.empty:
        continue

    datos_personalizados = np.column_stack([
        subconjunto["id"],
        subconjunto["nombre"],
        subconjunto["forma"],
        subconjunto["tipo_1"].astype(str),
        subconjunto["tipo_2"],
        subconjunto["generacion"],
        subconjunto["promedio_estadisticas"],
        subconjunto["sprite_url"],
    ])

    fig_final.add_trace(go.Scatter3d(
        x=subconjunto["generacion"],
        y=subconjunto["tipo_codigo"],
        z=subconjunto["promedio_estadisticas"],
        mode="markers",
        name=tipo,
        legendgroup=tipo,
        customdata=datos_personalizados,
        marker={
            "size": np.clip(subconjunto["promedio_estadisticas"] / 14, 3.5, 8.5),
            "color": colores_tipo[tipo],
            "opacity": 0.78,
            "line": {"width": 0.35, "color": "#222"},
        },
        hovertemplate=(
            "<b>%{customdata[1]}</b><br>"
            "Pokédex: #%{customdata[0]}<br>"
            "Forma: %{customdata[2]}<br>"
            "Tipo principal: %{customdata[3]}<br>"
            "Tipo secundario: %{customdata[4]}<br>"
            "Generación: %{customdata[5]}<br>"
            "Promedio: %{customdata[6]:.2f}"
            "<extra></extra>"
        )
    ))

fig_final.update_layout(
    title={
        "text": "Scatter Plot 3D de Pokémon con estadísticas, tipos y sprites",
        "x": 0.5,
        "xanchor": "center",
    },
    template="plotly_white",
    height=780,
    margin={"l": 0, "r": 0, "b": 0, "t": 70},
    legend={"title": "Tipo principal", "itemsizing": "constant"},
    scene={
        "xaxis": {"title": "Generación", "dtick": 1, "range": [0.7, 9.3]},
        "yaxis": {
            "title": "Tipo principal",
            "tickmode": "array",
            "tickvals": list(range(len(orden_tipos))),
            "ticktext": orden_tipos,
        },
        "zaxis": {"title": "Promedio de estadísticas", "range": [25, 125]},
        "camera": {"eye": {"x": 1.55, "y": 1.45, "z": 1.15}},
        "aspectmode": "manual",
        "aspectratio": {"x": 1.25, "y": 1.8, "z": 1.15},
    },
)

fig_final.show()

## 11. Sprites y filtros interactivos — 5 firmas

Plotly no permite insertar una imagen HTML directamente dentro del cuadro emergente de un punto 3D. Por ello, la integración se realiza mediante un **panel interactivo**: al pasar el cursor o hacer clic sobre un punto, se muestra el sprite, nombre, número, tipo, generación y promedio del Pokémon seleccionado.

El archivo final también incorpora filtros por:

- Generación.
- Tipo principal.
- Promedio mínimo y máximo de estadísticas.

In [20]:
import json
from pathlib import Path


def crear_html_interactivo_sprites(figura, tipos, ruta_salida):

    # =========================================================
    # 1. Preparar datos desde Python
    # =========================================================

    datos_originales = []

    for traza in figura.data:

        puntos = []

        for i in range(len(traza.x)):

            fila = traza.customdata[i]

            puntos.append({
                "x": float(traza.x[i]),
                "y": float(traza.y[i]),
                "z": float(traza.z[i]),

                "id": int(fila[0]),
                "nombre": str(fila[1]),
                "forma": str(fila[2]),
                "tipo1": str(fila[3]),
                "tipo2": str(fila[4]),
                "generacion": int(fila[5]),
                "promedio": float(fila[6]),
                "sprite": str(fila[7])
            })

        datos_originales.append({
            "name": str(traza.name),
            "puntos": puntos
        })


    datos_json = json.dumps(
        datos_originales,
        ensure_ascii=False
    )


    # =========================================================
    # 2. Crear HTML Plotly
    # =========================================================

    grafica_html = figura.to_html(
        full_html=False,
        include_plotlyjs=True,
        div_id="pokemonPlot3D",

        config={
            "responsive": True,
            "displaylogo": False,
            "scrollZoom": True
        }
    )


    opciones_tipos = "".join(
        f"<option value='{tipo}'>{tipo}</option>"
        for tipo in tipos
    )

    opciones_generacion = "".join(
        f"<option value='{g}'>{g}</option>"
        for g in range(1, 10)
    )


    # =========================================================
    # 3. HTML
    # =========================================================

    html_final = f"""
<!DOCTYPE html>

<html lang="es">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Pokémon 3D con sprites</title>


<style>

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    padding: 18px;
    font-family: Arial, sans-serif;
    background: #f5f7fb;
    color: #17233b;
}}


h1 {{
    text-align: center;
    color: #2a75bb;
}}


.panel-filtros {{

    display: grid;

    grid-template-columns:
        repeat(5, 1fr);

    gap: 10px;

    background: white;

    padding: 14px;

    border-radius: 14px;

    margin-bottom: 14px;
}}


label {{
    font-weight: bold;
}}


select,
input,
button {{

    width: 100%;

    margin-top: 5px;

    padding: 8px;

    border-radius: 8px;

    border: 1px solid #bbb;
}}


button {{

    background: #ffcb05;

    font-weight: bold;

    cursor: pointer;
}}


#contador {{

    grid-column: 1 / -1;

    text-align: center;

    font-size: 18px;

    font-weight: bold;

    color: #2a75bb;
}}


.contenido {{

    display: grid;

    grid-template-columns:
        minmax(0, 1fr)
        250px;

    gap: 15px;
}}


.contenedor-grafica {{

    position: relative;

    background: white;

    border-radius: 14px;

    overflow: hidden;
}}


#grafica-wrapper {{

    position: relative;

    width: 100%;
}}


/* =========================================================
   CAPA DE SPRITES
   ========================================================= */

#sprite-layer {{

    position: absolute;

    left: 0;
    top: 0;

    width: 100%;
    height: 100%;

    pointer-events: none;

    z-index: 50;

    overflow: hidden;
}}


.sprite-pokemon {{

    position: absolute;

    width: 42px;

    height: 42px;

    object-fit: contain;

    image-rendering: pixelated;

    transform:
        translate(-50%, -50%);

    pointer-events: auto;

    cursor: pointer;

    transition:
        width .12s,
        height .12s,
        opacity .12s;

    filter:
        drop-shadow(
            0 1px 2px rgba(0,0,0,.3)
        );
}}


.sprite-pokemon:hover {{

    width: 65px;

    height: 65px;

    z-index: 999;
}}


.ficha {{

    background: white;

    border-radius: 14px;

    padding: 15px;

    text-align: center;
}}


.ficha img {{

    width: 150px;

    height: 150px;

    object-fit: contain;

    image-rendering: pixelated;
}}


.ficha h2 {{

    color: #2a75bb;
}}


.dato {{

    background: #f3f6fb;

    border-radius: 8px;

    padding: 8px;

    margin: 8px 0;
}}


@media(max-width: 900px) {{

    .contenido {{
        grid-template-columns: 1fr;
    }}

    .panel-filtros {{
        grid-template-columns:
            1fr 1fr;
    }}
}}

</style>

</head>


<body>


<h1>
Scatter Plot 3D de Pokémon con sprites
</h1>


<div class="panel-filtros">


<label>

Generación

<select id="filtroGeneracion">

<option value="Todas">
Todas
</option>

{opciones_generacion}

</select>

</label>



<label>

Tipo

<select id="filtroTipo">

<option value="Todos">
Todos
</option>

{opciones_tipos}

</select>

</label>



<label>

Promedio mínimo

<input
    id="promedioMinimo"
    type="number"
    value="0">

</label>



<label>

Promedio máximo

<input
    id="promedioMaximo"
    type="number"
    value="150">

</label>



<label>

Restablecer

<button id="botonRestablecer">

Mostrar todos

</button>

</label>


<div id="contador">

Preparando sprites...

</div>


</div>



<div class="contenido">


<div class="contenedor-grafica">

<div id="grafica-wrapper">

{grafica_html}

<div id="sprite-layer"></div>

</div>

</div>



<aside class="ficha">


<p>
Pasa el cursor sobre un Pokémon.
</p>


<img
    id="spriteFicha"
    src="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png">


<h2 id="nombreFicha">
Pikachu
</h2>


<div class="dato"
     id="numeroFicha">

Pokédex #25

</div>


<div class="dato"
     id="tipoFicha">

Tipo: Electric

</div>


<div class="dato"
     id="generacionFicha">

Generación: 1

</div>


<div class="dato"
     id="promedioFicha">

Promedio: 53.33

</div>


</aside>


</div>



<script>

(function iniciar() {{

    const gd =
        document.getElementById(
            "pokemonPlot3D"
        );

    const layer =
        document.getElementById(
            "sprite-layer"
        );


    if (
        !gd ||
        !gd._fullLayout ||
        !gd._fullLayout.scene
    ) {{

        setTimeout(
            iniciar,
            150
        );

        return;
    }}


    const datos =
        {datos_json};


    let todosPuntos = [];


    datos.forEach(traza => {{

        traza.puntos.forEach(p => {{

            todosPuntos.push(p);

        }});

    }});


    // =====================================================
    // CONTROLES
    // =====================================================

    const filtroGeneracion =
        document.getElementById(
            "filtroGeneracion"
        );

    const filtroTipo =
        document.getElementById(
            "filtroTipo"
        );

    const promedioMinimo =
        document.getElementById(
            "promedioMinimo"
        );

    const promedioMaximo =
        document.getElementById(
            "promedioMaximo"
        );

    const contador =
        document.getElementById(
            "contador"
        );


    // =====================================================
    // SPRITES
    // =====================================================

    const sprites = [];


    todosPuntos.forEach((punto, indice) => {{

        const img =
            document.createElement(
                "img"
            );


        img.src =
            punto.sprite;


        img.className =
            "sprite-pokemon";


        img.dataset.index =
            indice;


        img.alt =
            punto.nombre;


        img.addEventListener(

            "mouseenter",

            () => actualizarFicha(
                punto
            )

        );


        img.addEventListener(

            "click",

            () => actualizarFicha(
                punto
            )

        );


        layer.appendChild(
            img
        );


        sprites.push(img);

    }});


    // =====================================================
    // FICHA
    // =====================================================

    function actualizarFicha(p) {{

        document.getElementById(
            "spriteFicha"
        ).src =
            p.sprite;


        document.getElementById(
            "nombreFicha"
        ).textContent =
            p.nombre;


        document.getElementById(
            "numeroFicha"
        ).textContent =
            `Pokédex #${{p.id}} · ${{p.forma}}`;


        document.getElementById(
            "tipoFicha"
        ).textContent =

            p.tipo2 ===
            "Sin tipo secundario"

            ?

            `Tipo: ${{p.tipo1}}`

            :

            `Tipo: ${{p.tipo1}} / ${{p.tipo2}}`;


        document.getElementById(
            "generacionFicha"
        ).textContent =
            `Generación: ${{p.generacion}}`;


        document.getElementById(
            "promedioFicha"
        ).textContent =
            `Promedio: ${{p.promedio.toFixed(2)}}`;

    }}


    function puntoDesdeCustomdata(fila) {{

        return {{
            id: Number(fila[0]),
            nombre: String(fila[1]),
            forma: String(fila[2]),
            tipo1: String(fila[3]),
            tipo2: String(fila[4]),
            generacion: Number(fila[5]),
            promedio: Number(fila[6]),
            sprite: String(fila[7])
        }};

    }}


    gd.on("plotly_hover", evento => {{
        const fila = evento.points?.[0]?.customdata;
        if (fila) actualizarFicha(puntoDesdeCustomdata(fila));
    }});

    gd.on("plotly_click", evento => {{
        const fila = evento.points?.[0]?.customdata;
        if (fila) actualizarFicha(puntoDesdeCustomdata(fila));
    }});


    // =====================================================
    // FILTRO
    // =====================================================

    function puntoVisible(p) {{

        const gen =
            filtroGeneracion.value;

        const tipo =
            filtroTipo.value;

        const min =
            Number(
                promedioMinimo.value || 0
            );

        const max =
            Number(
                promedioMaximo.value || 150
            );


        return (

            (
                gen === "Todas"
                ||
                String(p.generacion) === gen
            )

            &&

            (
                tipo === "Todos"
                ||
                p.tipo1 === tipo
            )

            &&

            p.promedio >= min

            &&

            p.promedio <= max

        );

    }}


    function aplicarFiltros() {{

        let cantidadVisible = 0;
        const actualizaciones = [];

        datos.forEach((traza, indice) => {{

            const puntosVisibles = traza.puntos.filter(puntoVisible);
            cantidadVisible += puntosVisibles.length;

            const customdata = puntosVisibles.map(p => [
                p.id, p.nombre, p.forma, p.tipo1, p.tipo2,
                p.generacion, p.promedio, p.sprite
            ]);

            actualizaciones.push(Plotly.restyle(gd, {{
                x: [puntosVisibles.map(p => p.x)],
                y: [puntosVisibles.map(p => p.y)],
                z: [puntosVisibles.map(p => p.z)],
                customdata: [customdata]
            }}, [indice]));

        }});

        contador.textContent =
            `${{cantidadVisible.toLocaleString("es-MX")}} Pokémon visibles`;

        Promise.all(actualizaciones).then(solicitarActualizacion);

    }}


    // =====================================================
    // PROYECCIÓN 3D -> 2D
    // =====================================================

    function proyectarPunto(
        punto,
        escena
    ) {{

        try {{

            const glplot =
                escena.glplot;


            if (
                !glplot ||
                !glplot.cameraParams
            ) {{

                return null;

            }}


            const camera =
                glplot.cameraParams;


            const fullScene =
                gd._fullLayout.scene;


            const xaxis =
                fullScene.xaxis;

            const yaxis =
                fullScene.yaxis;

            const zaxis =
                fullScene.zaxis;


            /*
            Convertimos valores reales
            a coordenadas internas
            de Plotly.
            */

            const dataScale =
                escena.dataScale ||
                [1, 1, 1];


            const vector = [

                xaxis.r2l(punto.x)
                    * dataScale[0],

                yaxis.r2l(punto.y)
                    * dataScale[1],

                zaxis.r2l(punto.z)
                    * dataScale[2],

                1
            ];


            /*
            Multiplicación por matrices
            utilizada por Plotly gl3d.
            */

            function multiplicar(
                matriz,
                vector
            ) {{

                const resultado =
                    [0, 0, 0, 0];


                for (
                    let fila = 0;
                    fila < 4;
                    fila++
                ) {{

                    resultado[fila] =

                        matriz[
                            fila
                        ] * vector[0]

                        +

                        matriz[
                            fila + 4
                        ] * vector[1]

                        +

                        matriz[
                            fila + 8
                        ] * vector[2]

                        +

                        matriz[
                            fila + 12
                        ] * vector[3];

                }}


                return resultado;

            }}


            let proyectado =
                vector;


            /*
            cameraParams normalmente
            contiene las matrices que
            Plotly utiliza internamente.
            */

            if (
                camera.model
            ) {{

                proyectado =
                    multiplicar(
                        camera.model,
                        proyectado
                    );

            }}


            if (
                camera.view
            ) {{

                proyectado =
                    multiplicar(
                        camera.view,
                        proyectado
                    );

            }}


            if (
                camera.projection
            ) {{

                proyectado =
                    multiplicar(
                        camera.projection,
                        proyectado
                    );

            }}


            if (
                !proyectado[3]
            ) {{

                return null;

            }}


            const rect =
                escena.container
                    .getBoundingClientRect();


            const x =

                (
                    0.5
                    +
                    0.5
                    *
                    proyectado[0]
                    /
                    proyectado[3]
                )

                * rect.width;


            const y =

                (
                    0.5
                    -
                    0.5
                    *
                    proyectado[1]
                    /
                    proyectado[3]
                )

                * rect.height;


            return {{
                x: x,
                y: y,
                ancho: rect.width,
                alto: rect.height
            }};


        }}

        catch(error) {{

            return null;

        }}

    }}


    // =====================================================
    // ACTUALIZAR POSICIONES
    // =====================================================

    function actualizarSprites() {{

        const escena =
            gd._fullLayout
              .scene
              ._scene;


        if (!escena) {{
            return;
        }}


        let visibles = 0;


        todosPuntos.forEach(
            (punto, indice) => {{

                const sprite =
                    sprites[indice];


                if (
                    !puntoVisible(punto)
                ) {{

                    sprite.style.display =
                        "none";

                    return;

                }}


                const pos =
                    proyectarPunto(
                        punto,
                        escena
                    );


                if (!pos) {{

                    sprite.style.display =
                        "none";

                    return;

                }}


                if (
                    pos.x < -30
                    ||
                    pos.y < -30
                    ||
                    pos.x >
                        pos.ancho + 30
                    ||
                    pos.y >
                        pos.alto + 30
                ) {{

                    sprite.style.display =
                        "none";

                    return;

                }}


                sprite.style.display =
                    "block";


                sprite.style.left =
                    `${{pos.x}}px`;


                sprite.style.top =
                    `${{pos.y}}px`;


                visibles++;

            }}
        );


    }}


    // =====================================================
    // ACTUALIZACIÓN SUAVE DURANTE MOVIMIENTO
    // =====================================================

    let framePendiente =
        false;


    function solicitarActualizacion() {{

        if (framePendiente) {{
            return;
        }}


        framePendiente =
            true;


        requestAnimationFrame(
            () => {{

                actualizarSprites();

                framePendiente =
                    false;

            }}
        );

    }}


    // Durante rotación
    gd.on(
        "plotly_relayouting",
        solicitarActualizacion
    );


    // Al terminar rotación
    gd.on(
        "plotly_relayout",
        solicitarActualizacion
    );


    // Cambios de tamaño
    window.addEventListener(
        "resize",
        solicitarActualizacion
    );


    // =====================================================
    // FILTROS
    // =====================================================

    [

        filtroGeneracion,
        filtroTipo,
        promedioMinimo,
        promedioMaximo

    ].forEach(control => {{

        control.addEventListener(
            "change",
            aplicarFiltros
        );

    }});


    document.getElementById(
        "botonRestablecer"
    ).addEventListener(

        "click",

        () => {{

            filtroGeneracion.value =
                "Todas";

            filtroTipo.value =
                "Todos";

            promedioMinimo.value =
                0;

            promedioMaximo.value =
                150;

            aplicarFiltros();

        }}

    );


    // =====================================================
    // INICIO
    // =====================================================

    setTimeout(
        aplicarFiltros,
        500
    );


}})();

</script>


</body>

</html>
"""


    Path(
        ruta_salida
    ).write_text(
        html_final,
        encoding="utf-8"
    )


    return (
        Path(ruta_salida).resolve(),
        html_final
    )



# =============================================================
# CREAR NUEVA VISUALIZACIÓN
# =============================================================

RUTA_HTML_SPRITES, contenido_html_sprites = (
    crear_html_interactivo_sprites(

        fig_final,

        orden_tipos,

        "pokemon_scatter_3d_sprites.html"

    )
)


print(
    "Visualización con sprites creada:"
)

print(
    RUTA_HTML_SPRITES
)

print(
    f"Tamaño: "
    f"{RUTA_HTML_SPRITES.stat().st_size / (1024**2):.2f} MB"
)

Visualización con sprites creada:
/Users/farias/UTXJ/ECBD_9A_IDGS_230389/ECBD_9AIDGS_PRACTICAS_230389/Practica07/pokemon_scatter_3d_sprites.html
Tamaño: 5.04 MB


In [21]:
archivo_html = Path("pokemon_scatter_3d_sprites.html").resolve()

print(f"Archivo listo para abrir en el navegador: {archivo_html}")
display(HTML(f"<a href='{archivo_html.name}' target='_blank'>Abrir visualización interactiva</a>"))

Archivo listo para abrir en el navegador: /Users/farias/UTXJ/ECBD_9A_IDGS_230389/ECBD_9AIDGS_PRACTICAS_230389/Practica07/pokemon_scatter_3d_sprites.html


In [22]:
# Vista previa del HTML exportado dentro del Notebook
from IPython.display import IFrame

IFrame(
    src=archivo_html.name,
    width="100%",
    height=850
)

## 12. Identificación e interpretación de patrones — 2 firmas

In [23]:
resumen_generacion = (
    pokemon_visualizacion.groupby("generacion")["promedio_estadisticas"]
    .agg(cantidad="count", media="mean", mediana="median", maximo="max")
    .round(2)
)

resumen_tipo = (
    pokemon_visualizacion.groupby("tipo_1", observed=True)["promedio_estadisticas"]
    .agg(cantidad="count", media="mean", mediana="median", maximo="max")
    .sort_values("media", ascending=False)
    .round(2)
)

valores_atipicos_altos = pokemon_visualizacion.nlargest(
    10, "promedio_estadisticas"
)[["id", "nombre", "forma", "tipo_1", "generacion", "promedio_estadisticas"]]

print("Promedio por generación:")
display(resumen_generacion)
print("Promedio por tipo principal:")
display(resumen_tipo)
print("Pokémon con promedios más altos:")
display(valores_atipicos_altos)

Promedio por generación:


,cantidad,media,mediana,maximo
generacion,,,,
1,151,67.94,67.50,113.33
2,100,67.86,69.17,113.33
3,135,67.29,68.33,113.33
4,107,74.26,80.00,120.00
5,155,70.90,74.17,113.33
6,72,71.55,75.00,113.33
7,88,75.86,80.00,113.33
8,97,73.41,80.00,116.67
9,120,76.50,81.59,111.67


Promedio por tipo principal:


,cantidad,media,mediana,maximo
tipo_1,,,,
Dragon,37,81.69,81.67,113.33
Steel,36,79.18,83.33,113.33
Dark,45,75.79,81.33,113.33
Psychic,60,74.45,78.33,113.33
Fire,65,74.28,78.33,113.33
Fighting,40,73.76,77.50,116.67
Rock,58,73.70,77.50,100.00
Ice,32,72.96,79.59,96.67
Fairy,29,72.91,77.00,116.67


Pokémon con promedios más altos:


,id,nombre,forma,tipo_1,generacion,promedio_estadisticas
402,493,Arceus,Forma base,Normal,4,120.00
858,889,Zamazenta,Crowned Shield,Fighting,8,116.67
904,888,Zacian,Crowned Sword,Fairy,8,116.67
859,890,Eternatus,Forma base,Poison,8,115.00
121,150,Mewtwo,Forma base,Psychic,1,113.33
173,250,Ho-oh,Forma base,Fire,2,113.33
223,249,Lugia,Forma base,Psychic,2,113.33
372,384,Rayquaza,Forma base,Dragon,3,113.33
420,484,Palkia,Forma base,Water,4,113.33
482,487,Giratina,Altered Forme,Ghost,4,113.33


### Hallazgos relevantes

1. **Las generaciones recientes concentran promedios elevados.** En los registros base, la generación 9 presenta uno de los promedios generales más altos, mientras que las generaciones 1–3 muestran medias menores. Esto no significa que todos los Pokémon nuevos sean más fuertes, sino que su distribución contiene más especies con estadísticas altas.

2. **Dragon y Steel forman agrupaciones altas en el eje Z.** Estos tipos presentan promedios de estadísticas superiores a la mayoría de los demás tipos. En contraste, Bug concentra muchos puntos en niveles bajos y medios, debido a la presencia de numerosas etapas evolutivas tempranas.

3. **Los valores atípicos superiores corresponden principalmente a Pokémon legendarios.** Arceus aparece alrededor de 120 puntos de promedio; Zacian y Zamazenta en sus formas coronadas superan 116. Estos puntos quedan claramente separados de la concentración principal, que se encuentra aproximadamente entre 55 y 90.

4. **Existen valores bajos muy visibles.** Sunkern y Blipbug tienen promedios cercanos a 30, por lo que aparecen aislados en la parte inferior del gráfico. Son casos útiles para contrastar el rango completo de las estadísticas.

## 13. Exportación y comprobaciones finales — 4 firmas

In [24]:
# Comprobaciones automáticas antes de entregar
comprobaciones = {
    "dataset_no_vacio": not pokemon.empty,
    "seis_estadisticas_disponibles": all(c in pokemon.columns for c in variables_estadisticas),
    "columna_promedio_creada": "promedio_estadisticas" in pokemon.columns,
    "generaciones_1_a_9": sorted(pokemon_visualizacion["generacion"].unique().tolist()) == list(range(1, 10)),
    "tipos_codificados": pokemon_visualizacion["tipo_codigo"].ge(0).all(),
    "sprites_validos": pokemon_visualizacion["sprite_url_valida"].all(),
    "un_punto_por_id": not pokemon_visualizacion["id"].duplicated().any(),
    "html_exportado": RUTA_HTML_SPRITES.exists(),
}

resultado_comprobaciones = pd.Series(comprobaciones, name="cumple")
display(resultado_comprobaciones.to_frame())

assert all(comprobaciones.values()), "Hay una o más comprobaciones pendientes."
print("✅ Todas las comprobaciones se superaron. El Notebook está listo para entregarse.")

,cumple
dataset_no_vacio,True
seis_estadisticas_disponibles,True
columna_promedio_creada,True
generaciones_1_a_9,True
tipos_codificados,True
sprites_validos,True
un_punto_por_id,True
html_exportado,True


✅ Todas las comprobaciones se superaron. El Notebook está listo para entregarse.


## 14. Conclusiones finales — 2 firmas

La práctica permitió transformar un conjunto de estadísticas de Pokémon en una visualización 3D interactiva y comprensible. Primero se inspeccionaron los datos, se normalizaron las columnas y categorías, y se verificó la ausencia de registros inválidos o duplicados exactos. Después se calculó el promedio de HP, ataque, defensa, ataque especial, defensa especial y velocidad como indicador general de rendimiento.

La gráfica final facilita la comparación simultánea entre generaciones, tipos principales y estadísticas. La separación por colores permite reconocer agrupaciones por tipo, mientras que los filtros reducen la visualización a subconjuntos específicos. El panel de sprites mejora la identificación de cada punto sin saturar el espacio tridimensional. Finalmente, la exportación a HTML conserva la rotación, el zoom, la información emergente, los filtros y la interacción con los sprites.

Los resultados muestran que los datos no se distribuyen de manera uniforme: ciertos tipos y generaciones concentran promedios más altos, y los Pokémon legendarios forman valores atípicos claramente identificables. Esto demuestra la utilidad de las visualizaciones interactivas para explorar conjuntos de datos con varias dimensiones.

## Lista de verificación de la rúbrica (50 firmas)

| Criterio | Firmas |
|---|---:|
| Portada | 2 |
| Librerías | 2 |
| Carga y descripción del dataset | 3 |
| Inspección inicial | 2 |
| Normalización de columnas y categorías | 3 |
| Nulos, duplicados y datos incorrectos | 2 |
| Selección y justificación de estadísticas | 3 |
| Columna `promedio_estadisticas` | 2 |
| Estadística descriptiva | 4 |
| Preparación de generación y tipo | 3 |
| Rutas de sprites | 2 |
| Primera gráfica 3D | 3 |
| Diferenciación por tipo | 3 |
| Información emergente | 2 |
| Integración de sprites | 3 |
| Filtros interactivos | 2 |
| Personalización de diseño | 3 |
| Hallazgos | 2 |
| Exportación HTML | 2 |
| Conclusiones y ejecución final | 2 |
| **Total** | **50** |

## Publicación en GitHub

**Repositorio:** [ECBD_9AIDGS_PRACTICAS_230389](https://github.com/fariasdgs/ECBD_9AIDGS_PRACTICAS_230389)

Antes de entregar:

1. Ejecutar **Kernel → Restart & Run All** y confirmar que no existan errores.
2. Verificar `pokemon_scatter_3d_sprites.html` en un navegador.
3. Subir la carpeta `Practica07` a la rama `practica08` del repositorio.